# VM: action-conditioned World Model and multi-step rollout

English | [简体中文](01_finetune.zh-CN.ipynb)

## Goal

Residual MLP for numerical state + action → next/multi-step states; not a video-generation world model.

Run validation → training → validation selection → test report → save/reload →
inference → epoch recovery → publication preview.

**All fixtures are synthetic and tiny weights start randomly. This verifies
engineering behavior, not real model quality.** The first backend supports only
single-process, single-device fp32, defaults to CPU and needs no model download.

## Setup

Locally install `python -m pip install -e '.[vm,notebooks]'`.
On Colab the next cell clones the repository and installs dependencies. Put real
data/checkpoints on persistent storage such as Google Drive before training.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    checkout = Path("/content/advai-notebooks")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/advai-x/advai-notebooks.git", str(checkout)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e",
                    f"{checkout}[vm,notebooks]"], check=True)
    os.chdir(checkout)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))
print("Repository ready; install .[vm,notebooks] locally if needed.")

Repository ready; install .[vm,notebooks] locally if needed.


## Steps — Configuration and data checks

Use a fresh output directory. Require independent train/validation/test splits;
source groups, related episodes and duplicate content must not cross splits.
VLA/VM fit normalization on training data only.

In [2]:
import json
from uuid import uuid4

from advai_notebooks.common.runtime import setup
from advai_notebooks.vm.config import load_config
from advai_notebooks.vm.data import load_data, prepare_data
from advai_notebooks.vm.model import VMModel
from advai_notebooks.vm.pipeline import run
from advai_notebooks.vm.evaluate import evaluate
from advai_notebooks.vm.publish import publish

cfg = load_config(ROOT / "configs/vm/smoke.yaml")
cfg.output_dir = f"outputs/vm/notebook-{uuid4().hex[:8]}"
# model.id can point to an existing VM export for fine-tuning.
model = VMModel(cfg.model, setup(cfg), cfg.root, cfg.training)
rows, summary = load_data(cfg)
prepared = prepare_data(rows, model, cfg, fit=True)
print(json.dumps({"data": summary, "prepared_rows": {k: len(v) for k, v in prepared.items()}}, indent=2))

{
  "data": {
    "rows": 6,
    "splits": {
      "train": 3,
      "validation": 2,
      "test": 1
    },
    "fingerprint": "34cfdcb5941578236cd9753953306aea1f39fa8848965cf98877b8784c88d1d6",
    "windows": 18
  },
  "prepared_rows": {
    "train": 9,
    "validation": 6,
    "test": 3
  }
}


In [3]:
example = next(row for row in rows if row["split"] == "test")
print({"observation": example["observation"], "actions": example["actions"], "mask": example["mask"]})

{'observation': [0.5, 1], 'actions': [[0.2, 0], [0, 0.1], [-0.1, 0.2]], 'mask': [1, 1, 1]}


### Train, validate and select

Selection minimizes validation recursive rollout MSE, including epoch-zero baseline. Test does
not select models. `export/` saves validation best; `latest_export/` saves the last
epoch. Runs retain baseline, epoch checkpoints, optimizer/RNG, environment and provenance.

In [4]:
report = run(cfg)
print(json.dumps({
    "baseline_test": report["baseline_test"]["metrics"],
    "selected_test": report["finetuned_test"]["metrics"],
    "selected_epoch": report["training"]["best_epoch"],
    "completed_epoch": report["training"]["epoch"],
}, indent=2))
assert report["training"]["epoch"] == cfg.training.epochs

{
  "baseline_test": {
    "state_mse": 0.03328674721221129,
    "rollout_mse": 0.05727585994948944
  },
  "selected_test": {
    "state_mse": 0.026992981632550556,
    "rollout_mse": 0.04444906363884608
  },
  "selected_epoch": 2,
  "completed_epoch": 2
}


## Checks — Native export, reload and inference

Reload the export, recompute held-out metrics and assert agreement with the training
report. Tiny generated text/actions/states demonstrate interfaces only and must
not be used for business decisions or robot execution.

In [5]:
loaded = VMModel.load(report["export"])
reloaded_metrics = evaluate(loaded, prepare_data(rows, loaded, cfg, fit=False), cfg, "test")
assert reloaded_metrics["metrics"] == report["finetuned_test"]["metrics"]
actions = example["actions"][:sum(example["mask"])]
prediction = loaded.rollout(example["observation"], actions)
assert len(prediction["observations"]) == len(actions)
assert loaded.predict(example["observation"], actions[0]) == prediction["observations"][0]
print(json.dumps(prediction, indent=2))

{
  "observations": [
    [
      0.3410988748073578,
      1.0077394247055054
    ],
    [
      0.27189183235168457,
      1.180297613143921
    ],
    [
      0.24666890501976013,
      1.4369919300079346
    ]
  ],
  "state_spec": {
    "observation_names": [
      "x",
      "y"
    ],
    "action_names": [
      "dx",
      "dy"
    ],
    "observation_units": "meters",
    "action_units": "meters",
    "control_frequency_hz": 10.0
  }
}


### Fine-tune an existing World Model

Start a new experiment from native VM weights, preserving original normalization
so learned state/action meanings are not redefined. Real applications should use
new grouped data with matching physical semantics.

In [6]:
import copy

tuning = copy.deepcopy(cfg)
tuning.model.id = report["export"]
tuning.output_dir = f"outputs/vm/notebook-finetune-{uuid4().hex[:8]}"
tuning.training.learning_rate = 0.0005
tuned_report = run(tuning)
tuned = VMModel.load(tuned_report["export"])
assert tuned.statistics == loaded.statistics
print({"finetuned_test": tuned_report["finetuned_test"]["metrics"]})

{'finetuned_test': {'state_mse': 0.02420012094080448, 'rollout_mse': 0.03897076938301325}}


### Strict epoch-boundary recovery

The next cell resumes an already completed run: it is idempotent and performs no
additional training. Interrupted runs use the same entry point and restart from
the latest completed epoch. Configuration, data, source and relevant dependencies
must remain unchanged. Regression tests cover exact interrupted-run recovery.

In [7]:
resumed = run(cfg, resume=cfg.path(cfg.output_dir) / "latest")
assert resumed["training"] == report["training"]
print({"resume_epoch": resumed["training"]["epoch"], "history_unchanged": True})

{'resume_epoch': 2, 'history_unchanged': True}


### Offline publication preview

Validate the inference-file whitelist and reload without Hub access, repository
creation or upload. Real publication requires explicitly disabling `dry_run` and
checking base licenses, data authorization and model cards.

In [8]:
plan = publish(report["export"], "example/vm-training-guide", dry_run=True)
assert plan["status"] == "dry_run"
names = [item["path"] for item in plan["files"]]
assert not any("training_state" in name for name in names)
print({"status": plan["status"], "inference_files": names})

{'status': 'dry_run', 'inference_files': ['README.md', 'advai_metadata.json', 'model.safetensors']}


## Next Steps

- [Full guide](../../docs/vm.md): fields, inference JSON, methods and limitations.
- [Configurations](../../configs/vm/README.md): smoke and supported base/fine-tuning templates.
- Validate smoke first. Real model-aware validation loads weights and needs enough hardware memory; use a new output directory for changed data/settings.
- Check errors at different horizons and out-of-distribution trajectories. Short synthetic sequences do not demonstrate planning. Fine-tune existing VM exports only with matching state/action semantics.

CPU smoke and export reload have been executed. CUDA/MPS, large real-model training
and real robotics/video capability are outside this notebook's validation scope.